In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Make the notebook runnable from either the repository root or this folder.
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "tinytorch").is_dir()
)
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Autograd experiments

## tl;dr

Every implemented backward rule and the `_sum_to_shape` helper is exercised directly:
a known `grad_output` is fed into `.apply(...)` and the returned gradients are asserted.
All 48 checks pass: 7 for the helper and 41 across the nine backward classes (Add, Sub, Mul, Div, MatMul, Sum, Mean, Reshape, Transpose).
Each check verifies numerical values with `np.allclose`, gradient shapes, `None`
gradients for constants and `requires_grad=False` operands, broadcasting reduction, and
- for `Sum`/`Mean` - every `axis` and `keepdims` combination.

## Context & Methods

The tests call each `Function` subclass in isolation: build the operands as `Tensor_CP`
objects (or plain constants), instantiate the backward class, and call `.apply(grad_output)`
with a hand-chosen upstream gradient. Expected gradients come from the closed-form derivative
of each operation, computed independently with NumPy or written out by hand.

### Key assumptions

- `grad_output` is supplied directly rather than produced by a full backward pass, so these
  checks target the local derivative rules only.
- Expected values use the mathematical derivative (product rule, quotient rule, and so on),
  independent of the implementation's own reduction code.
- A small `check(...)` harness records each assertion's pass/fail instead of aborting on the
  first failure, so the final tallies can count everything.
- Numerical comparisons use `np.allclose` (atol 1e-6); shapes are compared exactly.

In [3]:
import numpy as np

from tinytorch.tensor import Tensor_CP
from tinytorch.autograd import (
    _sum_to_shape,
    AddBackward,
    SubBackward,
    MulBackward,
    DivBackward,
    MatMulBackward,
    SumBackward,
    MeanBackward,
    ReshapeBackward,
    TransposeBackward,
)

RESULTS = []


def check(class_name, description, verify):
    """Run one verification closure, record pass/fail, and never abort the notebook."""
    try:
        verify()
    except Exception as error:
        RESULTS.append(
            {"class": class_name, "description": description, "passed": False, "detail": str(error)}
        )
        print(f"FAIL [{class_name}] {description}")
        for line in str(error).splitlines():
            print(f"      {line}")
        return False
    RESULTS.append(
        {"class": class_name, "description": description, "passed": True, "detail": ""}
    )
    print(f"PASS [{class_name}] {description}")
    return True


def assert_values(name, actual, expected, atol=1e-6):
    """Assert numerical closeness with np.allclose, reporting expected vs actual."""
    actual_array = np.asarray(actual, dtype=float)
    expected_array = np.asarray(expected, dtype=float)
    if actual_array.shape != expected_array.shape:
        raise AssertionError(
            f"{name}: shape mismatch before value check "
            f"(expected {expected_array.shape}, actual {actual_array.shape})"
        )
    if not np.allclose(actual_array, expected_array, atol=atol):
        raise AssertionError(
            f"{name}: values differ\n"
            f"  expected: {expected_array.tolist()}\n"
            f"  actual:   {actual_array.tolist()}"
        )


def assert_shape(name, actual, expected_shape):
    """Assert the exact shape of a gradient array."""
    actual_shape = np.asarray(actual).shape
    if tuple(actual_shape) != tuple(expected_shape):
        raise AssertionError(
            f"{name}: shape mismatch\n"
            f"  expected shape: {tuple(expected_shape)}\n"
            f"  actual shape:   {tuple(actual_shape)}"
        )


def assert_is_none(name, value):
    """Assert that a gradient is None (constant or requires_grad=False operand)."""
    if value is not None:
        raise AssertionError(f"{name}: expected None, actual {value!r}")


def assert_raises(exception_type, operation, name):
    """Assert that an operation raises exactly the expected exception family."""
    try:
        operation()
    except exception_type:
        return
    except Exception as error:
        raise AssertionError(
            f"{name}: expected {exception_type.__name__}, got {type(error).__name__} ({error})"
        )
    raise AssertionError(f"{name}: expected {exception_type.__name__}, nothing raised")

## `_sum_to_shape`

`_sum_to_shape` reverses NumPy broadcasting: it sums a gradient back down to a target shape,
first collapsing any extra leading axes, then summing (with `keepdims`) any axis the target
keeps at size 1. A non-uniform gradient `[[1, 2, 3], [4, 5, 6]]` is used so a wrong reduction
axis cannot pass by accident.

In [4]:
grad_2x3 = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)


def sum_to_row_vector():
    result = _sum_to_shape(grad_2x3, (3,))
    assert_shape("(2,3) -> (3,)", result, (3,))
    assert_values("(2,3) -> (3,)", result, [5, 7, 9])            # column sums


check("_sum_to_shape", "collapses the leading axis to (3,) via column sums", sum_to_row_vector)


def sum_to_keepdim_row():
    result = _sum_to_shape(grad_2x3, (1, 3))
    assert_shape("(2,3) -> (1,3)", result, (1, 3))
    assert_values("(2,3) -> (1,3)", result, [[5, 7, 9]])


check("_sum_to_shape", "reduces broadcast axis 0 with keepdims to (1,3)", sum_to_keepdim_row)


def sum_to_keepdim_col():
    result = _sum_to_shape(grad_2x3, (2, 1))
    assert_shape("(2,3) -> (2,1)", result, (2, 1))
    assert_values("(2,3) -> (2,1)", result, [[6], [15]])         # row sums


check("_sum_to_shape", "reduces broadcast axis 1 with keepdims to (2,1)", sum_to_keepdim_col)


def sum_to_scalar():
    result = _sum_to_shape(grad_2x3, ())
    assert_shape("(2,3) -> ()", result, ())
    assert_values("(2,3) -> ()", result, 21)


check("_sum_to_shape", "sums everything down to a scalar ()", sum_to_scalar)


def sum_to_same_shape():
    result = _sum_to_shape(grad_2x3, (2, 3))
    assert_shape("(2,3) -> (2,3)", result, (2, 3))
    assert_values("(2,3) -> (2,3)", result, grad_2x3)            # unchanged


check("_sum_to_shape", "leaves an already-matching shape unchanged", sum_to_same_shape)


grad_2x2x3 = np.array(
    [[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]], dtype=float
)


def sum_to_drop_leading():
    result = _sum_to_shape(grad_2x2x3, (2, 3))
    assert_shape("(2,2,3) -> (2,3)", result, (2, 3))
    assert_values("(2,2,3) -> (2,3)", result, [[8, 10, 12], [14, 16, 18]])


check("_sum_to_shape", "drops an extra leading axis (2,2,3) -> (2,3)", sum_to_drop_leading)


def sum_to_drop_and_keepdim():
    result = _sum_to_shape(grad_2x2x3, (1, 3))
    assert_shape("(2,2,3) -> (1,3)", result, (1, 3))
    assert_values("(2,2,3) -> (1,3)", result, [[22, 26, 30]])


check("_sum_to_shape", "drops the leading axis then keepdims-reduces to (1,3)", sum_to_drop_and_keepdim)

print("\n_sum_to_shape checks complete.")

PASS [_sum_to_shape] collapses the leading axis to (3,) via column sums
PASS [_sum_to_shape] reduces broadcast axis 0 with keepdims to (1,3)
PASS [_sum_to_shape] reduces broadcast axis 1 with keepdims to (2,1)
PASS [_sum_to_shape] sums everything down to a scalar ()
PASS [_sum_to_shape] leaves an already-matching shape unchanged
PASS [_sum_to_shape] drops an extra leading axis (2,2,3) -> (2,3)
PASS [_sum_to_shape] drops the leading axis then keepdims-reduces to (1,3)

_sum_to_shape checks complete.


## AddBackward

Addition passes the upstream gradient straight through to each operand, then reduces it back
to that operand's shape. Broadcasting a `(2, 3)` tensor against a `(3,)` tensor means the
smaller operand collects a summed gradient. Constants and `requires_grad=False` operands get `None`.

In [5]:
add_a = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)
add_b = Tensor_CP([10, 20, 30], requires_grad=True)             # (3,) broadcasts
add_grad = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)


def add_broadcasting():
    grad_a, grad_b = AddBackward(add_a, add_b).apply(add_grad)
    assert_shape("grad wrt a", grad_a, (2, 3))
    assert_values("grad wrt a", grad_a, add_grad)               # identity passthrough
    assert_shape("grad wrt b", grad_b, (3,))
    assert_values("grad wrt b", grad_b, [5, 7, 9])              # summed over broadcast axis


check("AddBackward", "broadcasts (2,3)+(3,) and sums grad back to each operand", add_broadcasting)


def add_scalar_tensor():
    scalar = Tensor_CP(2.0, requires_grad=True)                 # shape ()
    grad_a, grad_scalar = AddBackward(add_a, scalar).apply(add_grad)
    assert_values("grad wrt a", grad_a, add_grad)
    assert_shape("grad wrt scalar", grad_scalar, ())
    assert_values("grad wrt scalar", grad_scalar, 21)          # whole gradient summed


check("AddBackward", "sums the whole gradient for a scalar () operand", add_scalar_tensor)


def add_python_constant():
    grad_a, grad_const = AddBackward(add_a, 5).apply(add_grad)
    assert_values("grad wrt a", grad_a, add_grad)
    assert_is_none("grad wrt constant 5", grad_const)


check("AddBackward", "returns None for a python-scalar constant operand", add_python_constant)


def add_requires_grad_false():
    frozen = Tensor_CP([10, 20, 30], requires_grad=False)
    grad_a, grad_frozen = AddBackward(add_a, frozen).apply(add_grad)
    assert_values("grad wrt a", grad_a, add_grad)
    assert_is_none("grad wrt requires_grad=False tensor", grad_frozen)


check("AddBackward", "returns None for a requires_grad=False operand", add_requires_grad_false)

print("\nAddBackward checks complete.")

PASS [AddBackward] broadcasts (2,3)+(3,) and sums grad back to each operand
PASS [AddBackward] sums the whole gradient for a scalar () operand
PASS [AddBackward] returns None for a python-scalar constant operand
PASS [AddBackward] returns None for a requires_grad=False operand

AddBackward checks complete.


## SubBackward

Subtraction sends `+grad_output` to the minuend and `-grad_output` to the subtrahend, each
reduced back through broadcasting. Either side becomes `None` when it is a constant or has
`requires_grad=False`.

In [6]:
sub_a = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)
sub_b = Tensor_CP([10, 20, 30], requires_grad=True)             # (3,)
sub_grad = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)


def sub_broadcasting():
    grad_a, grad_b = SubBackward(sub_a, sub_b).apply(sub_grad)
    assert_shape("grad wrt a", grad_a, (2, 3))
    assert_values("grad wrt a", grad_a, sub_grad)              # +grad
    assert_shape("grad wrt b", grad_b, (3,))
    assert_values("grad wrt b", grad_b, [-5, -7, -9])          # -grad, then summed


check("SubBackward", "routes +grad to the minuend and -grad to the subtrahend", sub_broadcasting)


def sub_left_frozen():
    frozen = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=False)
    grad_left, grad_right = SubBackward(frozen, sub_b).apply(sub_grad)
    assert_is_none("grad wrt frozen minuend", grad_left)
    assert_values("grad wrt b", grad_right, [-5, -7, -9])


check("SubBackward", "returns None for a requires_grad=False minuend", sub_left_frozen)


def sub_constant_subtrahend():
    grad_a, grad_const = SubBackward(sub_a, 5).apply(sub_grad)
    assert_values("grad wrt a", grad_a, sub_grad)
    assert_is_none("grad wrt constant subtrahend", grad_const)


check("SubBackward", "returns None for a python-scalar subtrahend", sub_constant_subtrahend)

print("\nSubBackward checks complete.")

PASS [SubBackward] routes +grad to the minuend and -grad to the subtrahend
PASS [SubBackward] returns None for a requires_grad=False minuend
PASS [SubBackward] returns None for a python-scalar subtrahend

SubBackward checks complete.


## MulBackward

The product rule gives `d(a*b)/da = grad_output * b` and `d(a*b)/db = grad_output * a`, each
reduced back to the operand shape so broadcasting is undone. A distinct `grad_output` and
distinct operand values keep the two partials from coinciding.

In [7]:
mul_a = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)
mul_b = Tensor_CP([10, 20, 30], requires_grad=True)             # (3,)
mul_grad = np.array([[1, 1, 1], [2, 2, 2]], dtype=float)


def mul_broadcasting():
    grad_a, grad_b = MulBackward(mul_a, mul_b).apply(mul_grad)
    assert_shape("grad wrt a", grad_a, (2, 3))
    assert_values("grad wrt a", grad_a, mul_grad * mul_b.data)              # grad * b
    assert_shape("grad wrt b", grad_b, (3,))
    assert_values("grad wrt b", grad_b, (mul_grad * mul_a.data).sum(axis=0))  # sum(grad * a)


check("MulBackward", "applies the product rule with broadcasting", mul_broadcasting)


def mul_scalar_tensor():
    scalar = Tensor_CP(2.0, requires_grad=True)
    ones_grad = np.ones((2, 3))
    grad_a, grad_scalar = MulBackward(mul_a, scalar).apply(ones_grad)
    assert_values("grad wrt a", grad_a, ones_grad * 2.0)
    assert_shape("grad wrt scalar", grad_scalar, ())
    assert_values("grad wrt scalar", grad_scalar, mul_a.data.sum())        # sum(grad * a) = 21


check("MulBackward", "handles a scalar () operand via the product rule", mul_scalar_tensor)


def mul_python_constant():
    grad_a, grad_const = MulBackward(mul_a, 3).apply(mul_grad)
    assert_values("grad wrt a", grad_a, mul_grad * 3)
    assert_is_none("grad wrt constant 3", grad_const)


check("MulBackward", "returns None for a python-scalar constant factor", mul_python_constant)


def mul_requires_grad_false():
    frozen = Tensor_CP([10, 20, 30], requires_grad=False)
    grad_a, grad_frozen = MulBackward(mul_a, frozen).apply(mul_grad)
    assert_values("grad wrt a", grad_a, mul_grad * frozen.data)
    assert_is_none("grad wrt requires_grad=False factor", grad_frozen)


check("MulBackward", "returns None for a requires_grad=False factor", mul_requires_grad_false)

print("\nMulBackward checks complete.")

PASS [MulBackward] applies the product rule with broadcasting
PASS [MulBackward] handles a scalar () operand via the product rule
PASS [MulBackward] returns None for a python-scalar constant factor
PASS [MulBackward] returns None for a requires_grad=False factor

MulBackward checks complete.


## DivBackward

The quotient rule gives `d(a/b)/da = grad_output / b` and `d(a/b)/db = grad_output * (-a / b**2)`,
each reduced back to its operand shape. Expected values are computed straight from that formula
with NumPy, so a wrong sign or a missing square would be caught.

In [8]:
div_a = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)
div_b = Tensor_CP([10.0, 20.0, 40.0], requires_grad=True)       # (3,)
div_grad = np.ones((2, 3))


def div_broadcasting():
    grad_a, grad_b = DivBackward(div_a, div_b).apply(div_grad)
    expected_a = div_grad / div_b.data
    expected_b = (div_grad * (-div_a.data / div_b.data ** 2)).sum(axis=0)
    assert_shape("grad wrt a", grad_a, (2, 3))
    assert_values("grad wrt a", grad_a, expected_a)
    assert_shape("grad wrt b", grad_b, (3,))
    assert_values("grad wrt b", grad_b, expected_b)


check("DivBackward", "applies the quotient rule with broadcasting", div_broadcasting)


def div_constant_denominator():
    grad_a, grad_const = DivBackward(div_a, 2).apply(div_grad)
    assert_values("grad wrt a", grad_a, div_grad / 2)
    assert_is_none("grad wrt constant denominator", grad_const)


check("DivBackward", "returns None for a python-scalar denominator", div_constant_denominator)


def div_numerator_frozen():
    frozen = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=False)
    grad_left, grad_right = DivBackward(frozen, div_b).apply(div_grad)
    assert_is_none("grad wrt frozen numerator", grad_left)
    expected_b = (div_grad * (-frozen.data / div_b.data ** 2)).sum(axis=0)
    assert_values("grad wrt b", grad_right, expected_b)


check("DivBackward", "returns None for a requires_grad=False numerator", div_numerator_frozen)

print("\nDivBackward checks complete.")

PASS [DivBackward] applies the quotient rule with broadcasting
PASS [DivBackward] returns None for a python-scalar denominator
PASS [DivBackward] returns None for a requires_grad=False numerator

DivBackward checks complete.


## MatMulBackward (2D only)

For `C = L @ R`, the gradients are `grad_L = grad_output @ R.T` and `grad_R = L.T @ grad_output`.
Shapes are `(3, 2) @ (2, 4) -> (3, 4)`, so the two gradients must come back at `(3, 2)` and
`(2, 4)`. The class also validates the `grad_output` shape and should reject a wrong one.

In [9]:
mm_left = Tensor_CP([[1, 2], [3, 4], [5, 6]], requires_grad=True)         # (3, 2)
mm_right = Tensor_CP([[1, 0, 1, 0], [0, 1, 0, 1]], requires_grad=True)    # (2, 4)
mm_grad = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], dtype=float)  # (3, 4)


def matmul_both():
    grad_left, grad_right = MatMulBackward(mm_left, mm_right).apply(mm_grad)
    assert_shape("grad wrt left", grad_left, (3, 2))
    assert_values("grad wrt left", grad_left, mm_grad @ mm_right.data.T)
    assert_shape("grad wrt right", grad_right, (2, 4))
    assert_values("grad wrt right", grad_right, mm_left.data.T @ mm_grad)


check("MatMulBackward", "computes grad @ R.T and L.T @ grad for 2D operands", matmul_both)


def matmul_right_frozen():
    frozen_right = Tensor_CP([[1, 0, 1, 0], [0, 1, 0, 1]], requires_grad=False)
    grad_left, grad_right = MatMulBackward(mm_left, frozen_right).apply(mm_grad)
    assert_values("grad wrt left", grad_left, mm_grad @ frozen_right.data.T)
    assert_is_none("grad wrt requires_grad=False right", grad_right)


check("MatMulBackward", "returns None for a requires_grad=False right operand", matmul_right_frozen)


def matmul_left_frozen():
    frozen_left = Tensor_CP([[1, 2], [3, 4], [5, 6]], requires_grad=False)
    grad_left, grad_right = MatMulBackward(frozen_left, mm_right).apply(mm_grad)
    assert_is_none("grad wrt requires_grad=False left", grad_left)
    assert_values("grad wrt right", grad_right, frozen_left.data.T @ mm_grad)


check("MatMulBackward", "returns None for a requires_grad=False left operand", matmul_left_frozen)


def matmul_bad_grad_shape():
    assert_raises(
        ValueError,
        lambda: MatMulBackward(mm_left, mm_right).apply(np.ones((3, 3))),
        "wrong grad_output shape",
    )


check("MatMulBackward", "raises ValueError when grad_output shape is wrong", matmul_bad_grad_shape)

print("\nMatMulBackward checks complete.")

PASS [MatMulBackward] computes grad @ R.T and L.T @ grad for 2D operands
PASS [MatMulBackward] returns None for a requires_grad=False right operand
PASS [MatMulBackward] returns None for a requires_grad=False left operand
PASS [MatMulBackward] raises ValueError when grad_output shape is wrong

MatMulBackward checks complete.


## SumBackward

The gradient of a sum is the upstream gradient broadcast back to every input element. When the
forward pass dropped an axis (`keepdims=False`), that axis is restored with `np.expand_dims`
before broadcasting. This section covers `axis=None`, `axis=0`, `axis=1`, a tuple axis, both
`keepdims` settings, a `requires_grad=False` tensor, and a scalar tensor.

In [10]:
sum_x = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)


def sum_axis_none():
    grad_input, = SumBackward(sum_x).apply(np.array(2.0))
    assert_shape("axis=None grad", grad_input, (2, 3))
    assert_values("axis=None grad", grad_input, np.full((2, 3), 2.0))


check("SumBackward", "axis=None broadcasts the scalar grad over every element", sum_axis_none)


def sum_axis_0():
    grad_input, = SumBackward(sum_x, axis=0).apply(np.array([1.0, 2.0, 3.0]))
    assert_shape("axis=0 grad", grad_input, (2, 3))
    assert_values("axis=0 grad", grad_input, [[1, 2, 3], [1, 2, 3]])


check("SumBackward", "axis=0 keepdims=False expands then broadcasts down columns", sum_axis_0)


def sum_axis_1():
    grad_input, = SumBackward(sum_x, axis=1).apply(np.array([10.0, 20.0]))
    assert_shape("axis=1 grad", grad_input, (2, 3))
    assert_values("axis=1 grad", grad_input, [[10, 10, 10], [20, 20, 20]])


check("SumBackward", "axis=1 keepdims=False expands then broadcasts across rows", sum_axis_1)


def sum_axis_0_keepdims():
    grad_input, = SumBackward(sum_x, axis=0, keepdims=True).apply(np.array([[1.0, 2.0, 3.0]]))
    assert_shape("axis=0 keepdims grad", grad_input, (2, 3))
    assert_values("axis=0 keepdims grad", grad_input, [[1, 2, 3], [1, 2, 3]])


check("SumBackward", "axis=0 keepdims=True broadcasts the (1,3) grad directly", sum_axis_0_keepdims)


def sum_axis_1_keepdims():
    grad_input, = SumBackward(sum_x, axis=1, keepdims=True).apply(np.array([[10.0], [20.0]]))
    assert_shape("axis=1 keepdims grad", grad_input, (2, 3))
    assert_values("axis=1 keepdims grad", grad_input, [[10, 10, 10], [20, 20, 20]])


check("SumBackward", "axis=1 keepdims=True broadcasts the (2,1) grad directly", sum_axis_1_keepdims)


sum_y = Tensor_CP(np.arange(24).reshape(2, 3, 4), requires_grad=True)   # (2, 3, 4)


def sum_axis_tuple():
    grad_input, = SumBackward(sum_y, axis=(0, 2)).apply(np.array([1.0, 2.0, 3.0]))
    expected = np.broadcast_to(np.array([1.0, 2.0, 3.0]).reshape(1, 3, 1), (2, 3, 4))
    assert_shape("tuple-axis grad", grad_input, (2, 3, 4))
    assert_values("tuple-axis grad", grad_input, expected)


check("SumBackward", "tuple axis=(0,2) restores both reduced axes", sum_axis_tuple)


def sum_requires_grad_false():
    frozen = Tensor_CP([[1, 2, 3]], requires_grad=False)
    grad_input, = SumBackward(frozen).apply(np.array(1.0))
    assert_is_none("frozen sum grad", grad_input)


check("SumBackward", "returns (None,) for a requires_grad=False tensor", sum_requires_grad_false)


def sum_scalar_tensor():
    scalar = Tensor_CP(5.0, requires_grad=True)
    grad_input, = SumBackward(scalar).apply(np.array(1.0))
    assert_shape("scalar sum grad", grad_input, ())
    assert_values("scalar sum grad", grad_input, 1.0)


check("SumBackward", "handles a scalar () tensor (grad stays scalar 1.0)", sum_scalar_tensor)

print("\nSumBackward checks complete.")

PASS [SumBackward] axis=None broadcasts the scalar grad over every element
PASS [SumBackward] axis=0 keepdims=False expands then broadcasts down columns
PASS [SumBackward] axis=1 keepdims=False expands then broadcasts across rows
PASS [SumBackward] axis=0 keepdims=True broadcasts the (1,3) grad directly
PASS [SumBackward] axis=1 keepdims=True broadcasts the (2,1) grad directly
PASS [SumBackward] tuple axis=(0,2) restores both reduced axes
PASS [SumBackward] returns (None,) for a requires_grad=False tensor
PASS [SumBackward] handles a scalar () tensor (grad stays scalar 1.0)

SumBackward checks complete.


## MeanBackward

Mean behaves like sum but divides by the number of reduced elements: the total size for
`axis=None`, one dimension length for a single axis, or the product of the reduced dimensions
for a tuple axis. The same axis/`keepdims`/`requires_grad`/scalar matrix as `SumBackward` is used.

In [11]:
mean_x = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)


def mean_axis_none():
    grad_input, = MeanBackward(mean_x).apply(np.array(1.0))
    assert_shape("axis=None grad", grad_input, (2, 3))
    assert_values("axis=None grad", grad_input, np.full((2, 3), 1 / 6))   # divide by size 6


check("MeanBackward", "axis=None divides the broadcast grad by the total count", mean_axis_none)


def mean_axis_0():
    grad_input, = MeanBackward(mean_x, axis=0).apply(np.array([1.0, 2.0, 3.0]))
    assert_shape("axis=0 grad", grad_input, (2, 3))
    assert_values("axis=0 grad", grad_input, np.array([[1, 2, 3], [1, 2, 3]]) / 2)


check("MeanBackward", "axis=0 divides by the reduced dimension length (2)", mean_axis_0)


def mean_axis_1():
    grad_input, = MeanBackward(mean_x, axis=1).apply(np.array([10.0, 20.0]))
    assert_shape("axis=1 grad", grad_input, (2, 3))
    assert_values("axis=1 grad", grad_input, np.array([[10, 10, 10], [20, 20, 20]]) / 3)


check("MeanBackward", "axis=1 divides by the reduced dimension length (3)", mean_axis_1)


def mean_axis_0_keepdims():
    grad_input, = MeanBackward(mean_x, axis=0, keepdims=True).apply(np.array([[2.0, 4.0, 6.0]]))
    assert_shape("axis=0 keepdims grad", grad_input, (2, 3))
    assert_values("axis=0 keepdims grad", grad_input, np.array([[2, 4, 6], [2, 4, 6]]) / 2)


check("MeanBackward", "axis=0 keepdims=True broadcasts (1,3) grad and divides by 2", mean_axis_0_keepdims)


def mean_axis_1_keepdims():
    grad_input, = MeanBackward(mean_x, axis=1, keepdims=True).apply(np.array([[30.0], [60.0]]))
    assert_shape("axis=1 keepdims grad", grad_input, (2, 3))
    assert_values("axis=1 keepdims grad", grad_input, np.array([[30, 30, 30], [60, 60, 60]]) / 3)


check("MeanBackward", "axis=1 keepdims=True broadcasts (2,1) grad and divides by 3", mean_axis_1_keepdims)


mean_y = Tensor_CP(np.arange(24).reshape(2, 3, 4), requires_grad=True)   # (2, 3, 4)


def mean_axis_tuple():
    grad_input, = MeanBackward(mean_y, axis=(0, 2)).apply(np.array([1.0, 2.0, 3.0]))
    expected = np.broadcast_to(np.array([1.0, 2.0, 3.0]).reshape(1, 3, 1), (2, 3, 4)) / 8
    assert_shape("tuple-axis grad", grad_input, (2, 3, 4))
    assert_values("tuple-axis grad", grad_input, expected)                 # divide by 2*4 = 8


check("MeanBackward", "tuple axis=(0,2) divides by the product of reduced axes (8)", mean_axis_tuple)


def mean_requires_grad_false():
    frozen = Tensor_CP([[1, 2, 3]], requires_grad=False)
    grad_input, = MeanBackward(frozen).apply(np.array(1.0))
    assert_is_none("frozen mean grad", grad_input)


check("MeanBackward", "returns (None,) for a requires_grad=False tensor", mean_requires_grad_false)


def mean_scalar_tensor():
    scalar = Tensor_CP(5.0, requires_grad=True)
    grad_input, = MeanBackward(scalar).apply(np.array(1.0))
    assert_shape("scalar mean grad", grad_input, ())
    assert_values("scalar mean grad", grad_input, 1.0)                     # count 1


check("MeanBackward", "handles a scalar () tensor (count 1, grad stays 1.0)", mean_scalar_tensor)

print("\nMeanBackward checks complete.")

PASS [MeanBackward] axis=None divides the broadcast grad by the total count
PASS [MeanBackward] axis=0 divides by the reduced dimension length (2)
PASS [MeanBackward] axis=1 divides by the reduced dimension length (3)
PASS [MeanBackward] axis=0 keepdims=True broadcasts (1,3) grad and divides by 2
PASS [MeanBackward] axis=1 keepdims=True broadcasts (2,1) grad and divides by 3
PASS [MeanBackward] tuple axis=(0,2) divides by the product of reduced axes (8)
PASS [MeanBackward] returns (None,) for a requires_grad=False tensor
PASS [MeanBackward] handles a scalar () tensor (count 1, grad stays 1.0)

MeanBackward checks complete.


## ReshapeBackward

Reshape is a pure re-layout, so its backward simply reshapes the upstream gradient back to the
original tensor shape while preserving row-major order. It is checked from a flat `(6,)` grad,
from a `(3, 2)` grad, from a `(1, 1)` grad back to a scalar, and for a `requires_grad=False` tensor.

In [12]:
reshape_x = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)


def reshape_from_flat():
    grad_input, = ReshapeBackward(reshape_x).apply(np.array([1.0, 2, 3, 4, 5, 6]))
    assert_shape("from (6,)", grad_input, (2, 3))
    assert_values("from (6,)", grad_input, [[1, 2, 3], [4, 5, 6]])


check("ReshapeBackward", "reshapes a (6,) grad back to (2,3)", reshape_from_flat)


def reshape_from_other():
    grad_input, = ReshapeBackward(reshape_x).apply(np.array([[1.0, 2], [3, 4], [5, 6]]))
    assert_shape("from (3,2)", grad_input, (2, 3))
    assert_values("from (3,2)", grad_input, [[1, 2, 3], [4, 5, 6]])       # row-major


check("ReshapeBackward", "reshapes a (3,2) grad back to (2,3)", reshape_from_other)


def reshape_scalar():
    scalar = Tensor_CP(7.0, requires_grad=True)
    grad_input, = ReshapeBackward(scalar).apply(np.array([[5.0]]))
    assert_shape("from (1,1) to ()", grad_input, ())
    assert_values("from (1,1) to ()", grad_input, 5.0)


check("ReshapeBackward", "reshapes a (1,1) grad back to a scalar ()", reshape_scalar)


def reshape_requires_grad_false():
    frozen = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=False)
    grad_input, = ReshapeBackward(frozen).apply(np.array([1.0, 2, 3, 4, 5, 6]))
    assert_is_none("frozen reshape grad", grad_input)


check("ReshapeBackward", "returns (None,) for a requires_grad=False tensor", reshape_requires_grad_false)

print("\nReshapeBackward checks complete.")

PASS [ReshapeBackward] reshapes a (6,) grad back to (2,3)
PASS [ReshapeBackward] reshapes a (3,2) grad back to (2,3)
PASS [ReshapeBackward] reshapes a (1,1) grad back to a scalar ()
PASS [ReshapeBackward] returns (None,) for a requires_grad=False tensor

ReshapeBackward checks complete.


## TransposeBackward

The gradient of a transpose is the transpose of the gradient. A rectangular `(3, 2)` grad must
come back as `(2, 3)`, a square grad must equal `grad.T`, and a `requires_grad=False` tensor
yields `None`.

In [13]:
transpose_x = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=True)   # (2, 3)


def transpose_rectangular():
    grad_output = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)      # (3, 2)
    grad_input, = TransposeBackward(transpose_x).apply(grad_output)
    assert_shape("(3,2) -> (2,3)", grad_input, (2, 3))
    assert_values("(3,2) -> (2,3)", grad_input, [[1, 3, 5], [2, 4, 6]])


check("TransposeBackward", "transposes a (3,2) grad back to (2,3)", transpose_rectangular)


def transpose_square_roundtrip():
    square = Tensor_CP([[1, 2], [3, 4]], requires_grad=True)
    grad_output = np.array([[10, 20], [30, 40]], dtype=float)
    grad_input, = TransposeBackward(square).apply(grad_output)
    assert_shape("square grad", grad_input, (2, 2))
    assert_values("square grad", grad_input, grad_output.T)


check("TransposeBackward", "transposes a square grad (matches grad.T)", transpose_square_roundtrip)


def transpose_requires_grad_false():
    frozen = Tensor_CP([[1, 2, 3], [4, 5, 6]], requires_grad=False)
    grad_input, = TransposeBackward(frozen).apply(np.array([[1, 2], [3, 4], [5, 6]], dtype=float))
    assert_is_none("frozen transpose grad", grad_input)


check("TransposeBackward", "returns (None,) for a requires_grad=False tensor", transpose_requires_grad_false)

print("\nTransposeBackward checks complete.")

PASS [TransposeBackward] transposes a (3,2) grad back to (2,3)
PASS [TransposeBackward] transposes a square grad (matches grad.T)
PASS [TransposeBackward] returns (None,) for a requires_grad=False tensor

TransposeBackward checks complete.


## Overall pass/fail summary

The tally below is generated from every `check(...)` recorded above: a per-component breakdown,
the overall pass/fail counts, and the details of any failure (exact test, expected, actual).

In [14]:
class_order = [
    "_sum_to_shape",
    "AddBackward",
    "SubBackward",
    "MulBackward",
    "DivBackward",
    "MatMulBackward",
    "SumBackward",
    "MeanBackward",
    "ReshapeBackward",
    "TransposeBackward",
]

print(f"{'component':<20} | {'passed':>6} | {'failed':>6} |  status")
print("-" * 52)
total_passed = 0
total_failed = 0
for class_name in class_order:
    class_results = [r for r in RESULTS if r["class"] == class_name]
    passed = sum(r["passed"] for r in class_results)
    failed = len(class_results) - passed
    total_passed += passed
    total_failed += failed
    status = "OK" if failed == 0 else "FAILED"
    print(f"{class_name:<20} | {passed:>6} | {failed:>6} |  {status}")

print("-" * 52)
print(f"{'TOTAL':<20} | {total_passed:>6} | {total_failed:>6} |")

failures = [r for r in RESULTS if not r["passed"]]
if failures:
    print("\nFailure details:")
    for r in failures:
        print(f"- [{r['class']}] {r['description']}")
        for line in r["detail"].splitlines():
            print(f"    {line}")

print(f"\nOverall: {total_passed} passed, {total_failed} failed out of {len(RESULTS)} checks.")
assert total_failed == 0, f"{total_failed} check(s) failed - see the details above."

component            | passed | failed |  status
----------------------------------------------------
_sum_to_shape        |      7 |      0 |  OK
AddBackward          |      4 |      0 |  OK
SubBackward          |      3 |      0 |  OK
MulBackward          |      4 |      0 |  OK
DivBackward          |      3 |      0 |  OK
MatMulBackward       |      4 |      0 |  OK
SumBackward          |      8 |      0 |  OK
MeanBackward         |      8 |      0 |  OK
ReshapeBackward      |      4 |      0 |  OK
TransposeBackward    |      3 |      0 |  OK
----------------------------------------------------
TOTAL                |     48 |      0 |

Overall: 48 passed, 0 failed out of 48 checks.
